### Types of Partitioning in Spark
<ul>
    Partition is generally used when there is a high cardinality
    <li>Hash Partitioning</li>
    <li>Range Partitioning</li>
</ul>

#### Hash Partitioning
<ul>
    <li>Hash Partitioning attempts to spread the data evenly across various partitions based
on the key.</li>
    <li> Object.hashCode method is used to determine the partition in Spark as
partition = key.hashCode ( ) % numPartitions.</li>
</ul>

#### Range Partitioning
<ul>
    <li>Some Spark RDDs have keys that follow a particular ordering for such RDDs range
partitioning is an efficient partitioning technique.</li>
    <li>In range partitioning method, tuples having keys within the same range will appear on
the same machine.</li>
    <li>Keys in a range partitioner are partitioned based on the set of sorted range of keys and
ordering of keys.</li>
</ul>

### When are we supposed to Use Partitioning and bucketting 

#### When to Use Partitioning

- You frequently **filter** by a column (e.g., `date`, `region`)
- The column has **low to medium cardinality**
- You want to **reduce scan size** and leverage **partition pruning**
- You're storing data in **S3 or HDFS**, where folder-based layout helps
- Your queries are **time-based**, like daily or monthly reports


#### When to Use Bucketing

- You frequently **join** or **groupBy** on a column (e.g., `user_ID`, `session_ID`)
- The column has **high cardinality**
- You want to **avoid shuffle** during joins or aggregations
- Both sides of a join are bucketed on the same column
- You're writing to a **Hive-compatible table** using `.saveAsTable()`


| Feature         | Partitioning                            | Bucketing                                 |
|----------------|------------------------------------------|-------------------------------------------|
| Storage Layout | Creates folders per partition column     | Stores bucketed files inside table folder |
| Optimization   | Enables partition pruning                | Enables shuffle avoidance and bucket pruning |
| Column Type    | Best for low-cardinality columns         | Best for high-cardinality columns         |
| Use Case       | Filter-heavy queries                     | Join-heavy or groupBy-heavy queries       |
| Compatibility  | Works well with S3/HDFS                  | Requires Hive-compatible table metadata   |

### UseCase 

for this example were the data is directly stored in s3 what are the best way to optimize in the below scenario
Your company collects millions of web server logs per day. Each log contains a timestamp, user ID, URL accessed, response time, and HTTP status code. Management wants to identify users who frequently encounter slow responses (>2s) or errors (5xx status codes), and generate a daily alert. Task: Explain how you would design a Spark job to process this data efficiently. Show pseudo code to calculate: Number of slow requests per user. Number of error requests per user. Explain how you would optimize this job for very large datasets (e.g., 100 million logs/day).





In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("WebLogAlertJob").getOrCreate()

# Read logs from S3 (Parquet recommended)
logs_df = spark.read.parquet("s3://web-logs/2025-09-24/") \
    .select("timestamp", "user_ID", "response_time", "HTTP_status_code")

# Filter slow responses and errors
filtered_df = logs_df.filter(
    (F.col("response_time") > 2) | (F.col("HTTP_status_code").between(500, 599))
)

# Aggregate per user
agg_df = filtered_df.groupBy("user_ID").agg(
    F.count("*").alias("total_flagged_requests"),
    F.sum(F.when(F.col("response_time") > 2, 1).otherwise(0)).alias("slow_requests"),
    F.sum(F.when(F.col("HTTP_status_code").between(500, 599), 1).otherwise(0)).alias("error_requests")
)

# Thresholding (optional)
alert_df = agg_df.filter((F.col("slow_requests") > 10) | (F.col("error_requests") > 5))

# Write alerts to S3
alert_df.write.mode("overwrite").parquet("s3://alerts/2025-09-24/")

In [ ]:
## Adding partitioning to the data for optmization

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

spark.conf.set("spark.sql.shuffle.partitions", "400")  # Based on cluster size


alert_df.write.partitionBy("date").parquet("s3://alerts/")


In [ ]:
## bucketting too 

df.write \
  .partitionBy("date") \
  .bucketBy(32, "user_ID") \
  .sortBy("user_ID") \
  .mode("overwrite") \
  .saveAsTable("bucketed_partitioned_logs")


## Note :  Bucketing only works when writing to a Hive-compatible table via .saveAsTable(). Glue datacatalog
## It won’t work with plain .parquet() writes to S3.




Since you're storing logs in S3 and processing daily alerts:
- ✅ Use partitioning by date for raw S3 storage
- ✅ Use bucketing by user_ID only if you're writing to a managed Hive table or Delta Lake
